# Prepare Training Data — AI Therapist

Downloads Hugging Face psychology datasets, converts them to **Qwen chat format**, and writes merged files to Google Drive.

## Sources
- **SFT:** `LuangMV97/Empathetic_counseling_Dataset`, `UKPLab/Graph2Counsel`, `psychologie-et-serenite/articles-metadata` (+ public API)
- **Eval only:** `AI-companionship/INTIMA` (prompts without labels)

## Outputs (on Drive)
- `clinical_synthetic_data.json` — merged training rows
- `intima_eval.json` — evaluation prompts
- `dataset_stats.json` — row counts per source

**Runtime:** CPU is fine. Article fetching takes ~10–15 min (923 articles @ 0.5s each).

## 1. Install dependencies

In [ ]:
!pip install -q "datasets>=2.19.0" "transformers>=4.44.0" "requests>=2.31.0"

## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/ai-therapist"
!mkdir -p "{PROJECT_DIR}"
print(f"Output dir: {PROJECT_DIR}")

## 3. Get repo scripts

Downloads `prepare_datasets.py` directly from GitHub (fast, no `git clone` auth prompts).

**If the previous cell (`drive.mount`) looks stuck:** click the Google authorization link in the output and approve access — it waits for you.

In [ ]:
import sys
import urllib.request
from pathlib import Path

# Public repo — raw download avoids git auth hangs in Colab
GITHUB_USER = "icors2"
GITHUB_REPO = "Therapist-model"
GITHUB_BRANCH = "main"
RAW_SCRIPT_URL = (
    f"https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/"
    f"{GITHUB_BRANCH}/scripts/prepare_datasets.py"
)

SCRIPT_DIR = Path("/content/scripts")
SCRIPT_PATH = SCRIPT_DIR / "prepare_datasets.py"
SCRIPT_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_SCRIPT_CANDIDATES = [
    Path("/content/drive/MyDrive/Therapist model/scripts/prepare_datasets.py"),
    Path("/content/drive/MyDrive/therapist-model/scripts/prepare_datasets.py"),
]

loaded = False
for candidate in DRIVE_SCRIPT_CANDIDATES:
    if candidate.exists():
        SCRIPT_PATH = candidate
        loaded = True
        print(f"Using script from Drive: {SCRIPT_PATH}")
        break

if not loaded:
    print(f"Downloading script from GitHub...\n  {RAW_SCRIPT_URL}")
    try:
        urllib.request.urlretrieve(RAW_SCRIPT_URL, SCRIPT_PATH)
        print(f"Saved to {SCRIPT_PATH}")
    except Exception as e:
        raise RuntimeError(
            "Could not download prepare_datasets.py. Check that the repo is public "
            f"and pushed to branch '{GITHUB_BRANCH}'.\n"
            f"URL: {RAW_SCRIPT_URL}\nError: {e}"
        ) from e

sys.path.insert(0, str(SCRIPT_PATH.parent))
from prepare_datasets import prepare_all

print("Ready.")

## 4. Configuration

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B"
MAX_EMPATHETIC = 10_000   # subsample for first Colab run
ARTICLE_MAX_CHARS = 1500
ARTICLE_RATE_LIMIT = 0.5  # seconds between API calls
SKIP_ARTICLES = True      # False = +~8 min fetching 923 articles from API

## 5. Download, convert, and save

**This cell takes several minutes** — progress lines print every 1000 Empathetic rows. Do not interrupt unless it has been idle with no new output for 10+ minutes.

Expected runtime (CPU): ~5–10 min with `SKIP_ARTICLES=True`, ~15–25 min with articles.

In [ ]:
stats = prepare_all(
    project_dir=PROJECT_DIR,
    model_id=MODEL_ID,
    max_empathetic=MAX_EMPATHETIC,
    article_max_chars=ARTICLE_MAX_CHARS,
    article_rate_limit=ARTICLE_RATE_LIMIT,
    skip_articles=SKIP_ARTICLES,
)

## 6. Preview output

In [ ]:
import json

data_path = f"{PROJECT_DIR}/clinical_synthetic_data.json"
with open(data_path, encoding="utf-8") as f:
    rows = json.load(f)

print(f"Total SFT rows: {len(rows)}")
print("\n--- First example (truncated) ---")
print(rows[0]["text"][:600], "...")

---

**Next:** Open [`ai_therapist_qlora_training.ipynb`](./ai_therapist_qlora_training.ipynb), set `USE_SAMPLE_DATA = False`, and run training on the prepared JSON.